# EDA Parte B: Historial de pagos (`PAY_*`, `BILL_AMT*`, `PAY_AMT*`) y correlaciones

**Autora:** Olena (`T02` — rama `olena/eda-historial`)  
**Objetivo:** Analizar el comportamiento histórico de los clientes durante los 6 meses previos (abril a septiembre de 2005) y su relación con el límite de crédito (`LIMIT_BAL`), así como detectar problemas de calidad o estructura (asimetría, valores negativos, multicolinealidad) para la fase de preprocesamiento.

## 1. Carga de datos y comprobación general
Cargamos el dataset original y verificamos sus dimensiones, tipos de datos, valores nulos y filas duplicadas.

In [ ]:
# Importamos las librerías necesarias para análisis y visualización
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración visual de los gráficos
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

# Cargamos el dataset desde la carpeta data/
df = pd.read_csv("../data/UCI_Credit_Card.csv")

# Comprobamos dimensiones, nulos y duplicados
print(f"Dimensiones del dataset (filas, columnas): {df.shape}")
print(f"Total de valores nulos: {df.isna().sum().sum()}")
print(f"Filas duplicadas: {df.duplicated().sum()}")

# Mostramos las primeras 5 filas de las columnas de historial y el target
cols_historial = (
    ["LIMIT_BAL"]
    + [c for c in df.columns if c.startswith(("PAY_", "BILL_AMT", "PAY_AMT"))]
)
df[cols_historial].head()

## 2. Estado de pago mensual (`PAY_0`, `PAY_2` a `PAY_6`)
Analizamos la distribución de los códigos de estado de pago y cómo varía el límite de crédito medio (`LIMIT_BAL`) según los meses de retraso.

- **Nota sobre nombres:** `PAY_0` corresponde a septiembre de 2005 (mes 1); no existe columna `PAY_1`. En el preprocesamiento se renombrará `PAY_0` -> `PAY_1`.
- **Interpretación de códigos:** `-2` = sin consumo, `-1` = pagado completo al día, `0` = uso de crédito rotativo (pago mínimo al día), `1` a `8` = meses de retraso acumulado.

In [ ]:
# Columnas de estado de pago (de septiembre a abril de 2005)
cols_pay = ["PAY_0", "PAY_2", "PAY_3", "PAY_4", "PAY_5", "PAY_6"]

# Tabla de frecuencias de cada código en los 6 meses
conteos_pay = df[cols_pay].apply(lambda s: s.value_counts()).fillna(0).astype(int)
print("Conteo de clientes por código de estado de pago en cada mes:")
display(conteos_pay)

# Relación entre PAY_0 (septiembre) y el límite de crédito (LIMIT_BAL)
resumen_pay0 = (
    df.groupby("PAY_0")["LIMIT_BAL"]
    .agg(clientes="count", limite_medio="mean", limite_mediano="median")
    .round(0)
)
print("\nLímite de crédito según el estado de pago en septiembre (PAY_0):")
display(resumen_pay0)

In [ ]:
# Gráfico: conteo de clientes y límite medio según estado de pago en septiembre (PAY_0)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1) Distribución de clientes por valor de PAY_0
sns.countplot(data=df, x="PAY_0", ax=axes[0], color="steelblue")
axes[0].set_title("Número de clientes según estado de pago (PAY_0 - Septiembre)")
axes[0].set_xlabel("Código de estado de pago (PAY_0)")
axes[0].set_ylabel("Número de clientes")

# 2) Límite de crédito medio por valor de PAY_0
sns.barplot(data=df, x="PAY_0", y="LIMIT_BAL", estimator=np.mean, errorbar=None, ax=axes[1], color="teal")
axes[1].set_title("Límite de crédito medio (LIMIT_BAL) según PAY_0")
axes[1].set_xlabel("Código de estado de pago (PAY_0)")
axes[1].set_ylabel("Límite de crédito medio (NT$)")

plt.tight_layout()
plt.show()